# 2. Data curation: activation-loop filters <a id="2"></a>
In this section we will curate our kinase dataset for input into conformational analysis.


## Table of contents

- [2.1 Kinase taxonomy](#13)
- [2.2 Filtering for activation loop](#22)
  - [2.3.1 Gap-length filter](#231gap)
- [2.3 Fixed-bounds activation-loop length filter](#23bounds)


## Backend map

How this notebook connects to `workflow/` modules:

```mermaid
flowchart LR
  nb["03-ActivationLoopFilters"]
  m0["workflow.ca_stripper"]
  nb --> m0
  m1["workflow.kinaseGroupLabelling"]
  nb --> m1
  m2["workflow.reconstruct"]
  nb --> m2
  m3["workflow.utilities"]
  nb --> m3
```


Our data curation pipeline is subdivided in the following sections: 
2. [Data curation](#2)   
    2.4. [Kinase taxonomy](#24)   
    2.5. [Filtering for activation loop](#25)   
    2.6. [Gap-length filter](#26)
    2.7. [Fixed-bounds activation-loop length filter](#27)

To get started, let's load some packages!

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from IPython.display import display, HTML

from workflow.ca_stripper import OutlierStripper
from workflow.kinaseGroupLabelling import KinaseGroupLabeller
from workflow.reconstruct import ProteinReconstructor
from workflow.utilities import PDBDownloader
from workflow.utilities import count_pdb_files, braf_res, clear_and_make, make_seg, copy_filtered_pdbs, copy_motif_filtered_datasets


## 2.1 Kinase taxonomy <a id="13"></a>
Annotate each extracted chain in `Results/InterPro_protein_chains/` (`PDB_CHAIN.pdb`) with UniProt / KinHub Manning metadata and UniProt CAUTION pseudokinase flags. Writes `Results/kinase_annotation_all_chains.csv` and `Results/excluded_pseudokinase_basenames.txt`, then shows kinome-group and species %-bar charts plus a pseudokinase pie.


In [ ]:
from workflow.kinaseGroupLabelling import KinaseGroupLabeller

lab = KinaseGroupLabeller()
annot_all = lab.annotate_dataset_chains_with_kinome(
    "Results/InterPro_protein_chains/",
    output_csv="Results/kinase_annotation_all_chains.csv",
)
display(annot_all.head())

figs = lab.plot_annotation_summary(annot_all, species_top_n=15, save_dir="Results")
for key in ("fig_group", "fig_species", "fig_pseudokinase"):
    display(figs[key])


As expected, our dataset includes kinase structures from all different kinase families.

## 2.2 Filtering for activation loop  <a id="22"></a>
Here we exclude all kinase domains that do not have the characteristic conserved residue motifs DFG and APE that delimit the activation loop.

We utilise `copy_motif_filtered_datasets()` to keep chains that contain DFG and APE (excluding pseudokinases), writing protein-only PDBs to `Results/motif_filtered_chains/` and mirroring the same basenames from the small-molecule extraction into `Results/motif_filtered_small_molecules/`.

In [ ]:
valid_pdbs, invalid_pdbs = copy_motif_filtered_datasets(
    source_dir_protein="Results/InterPro_protein_chains/",
    target_dir_protein="Results/motif_filtered_chains/",
    source_dir_small_molecules="Results/InterPro_protein_small_molecules/",
    target_dir_small_molecules="Results/motif_filtered_small_molecules/",
    excluded_basenames_file="Results/excluded_pseudokinase_basenames.txt",
)


Let's check how many motif-filtered protein chains and protein–small-molecule complexes we are left with.


In [ ]:
pdb_directory = 'Results/motif_filtered_chains/'
pdb_directory2 = 'Results/motif_filtered_small_molecules/'
pdb_count = count_pdb_files(pdb_directory)
pdb_count2 = count_pdb_files(pdb_directory2)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")
print(f"There are {pdb_count2} PDB files in the directory '{pdb_directory2}'.")


### 2.3.1 Gap-length filter  <a id="231gap"></a>
Crystal structures sometimes have missing residues in the activation loop that cannot be resolved by MODELLER. Chains are excluded here if the DFG–APE loop has more than **4 consecutive missing residues** **or** more than **7 missing residues in total** (input: `Results/motif_filtered_chains/`), before the loop-length filter in §2.3 and MODELLER in Workflow 2. Passing chains are written to `Results/gap_filtered_chains/` and a list of excluded basenames is saved as `Results/gap_filtered_chains/gap_excluded.txt`.


In [ ]:
from workflow.reconstruct import ProteinReconstructor

gap_filter = ProteinReconstructor(
    input_dir="Results/motif_filtered_chains/",
    full_pdb_dir="Results/InterPro_PDBs/",
    output_dir="Results/gap_filtered_chains/",
    max_gap_length=4,
    max_missing_residues=7,
)

gap_results = gap_filter.filter_by_max_gap()


## 2.3 Fixed-bounds activation-loop length filter  <a id="23bounds"></a>
We exclude structures whose **activation-loop sequence length** (SEQRES DFG→APE inclusive, counting missing residues) falls outside a fixed inclusive window (here 18–32 residues). On each run the loop TSV is rebuilt from full PDBs in `Results/InterPro_PDBs/` for the chains in `Results/gap_filtered_chains/`, then written to `Results/activation_loop_sequences.tsv`. PDBs that pass are written to `Results/Bounds_CAfilter_chains/`.


In [ ]:
from workflow.ca_stripper import OutlierStripper

INPUT_CHAINS_DIR = "Results/gap_filtered_chains/"
OUTPUT_CHAINS_DIR = "Results/Bounds_CAfilter_chains/"
FULL_PDB_DIR = "Results/InterPro_PDBs/"

length_filter_results = OutlierStripper(verbose=False).filter_chains_by_loop_length(
    input_chains_dir=INPUT_CHAINS_DIR,
    output_chains_dir=OUTPUT_CHAINS_DIR,
    full_pdb_dir=FULL_PDB_DIR,
    length_lower=18,
    length_upper=32,
)

Let's check how many kinase domains we are left with.

In [ ]:
pdb_directory = 'Results/Bounds_CAfilter_chains/'
pdb_count = count_pdb_files(pdb_directory)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")